# TOX3GNN – Optuna + 30-Run  Experiment
 **T4 GPU**

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Edit this to match where your TOX folder lives in Drive ──────
DRIVE_ROOT = '/content/drive/MyDrive/AUA/Thesis/code/HybridGNN/TOX'

os.makedirs(f'{DRIVE_ROOT}/checkpoints_tox21', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results_tox21',     exist_ok=True)

print('Drive mounted. Root:', DRIVE_ROOT)
print('Contents:', os.listdir(DRIVE_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Root: /content/drive/MyDrive/AUA/Thesis/code/HybridGNN/TOX
Contents: ['tox21_dataset (1).csv', 'tox21_dataset.csv', 'TOX3GNN.py', 'optim_TOX3GNN.py', 'results_tox21', 'feat_corr.py', '__pycache__', 'tox21_analysis.ipynb', 'imputation_checkpoints', 'tox21_imputed.csv', 'utils.py', 'TOX3GNN_experiment.ipynb', 'checkpoints_tox21']


## 2 · Install dependencies

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit', 'optuna'],  check=True)

import torch
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}')
print('Dependencies ready.')


torch=2.11.0+cu128  cuda=True
Dependencies ready.


## 3 · Imports

In [ ]:
import os, sys, time, random, shutil, warnings, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import optuna

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Linear

from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, GATConv, GINConv, SAGEConv
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from rdkit import Chem

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


Device: cuda


## 4 · Load `utils.py` from Drive
This imports your **exact** implementations — nothing is rewritten here.

In [ ]:
# utils.py must be in DRIVE_ROOT alongside tox21_dataset.csv
sys.path.insert(0, DRIVE_ROOT)

from utils import (
    one_hot_encoding,
    get_atom_features,
    get_bond_features,
    create_pytorch_geometric_graph_data_list_from_smiles_and_labels,
    scaffold_split,
    save_ckp,
    load_ckp,
    optimizer_to,
    round_to_4,
)

# Alias used throughout this notebook
smiles_to_graph_list = create_pytorch_geometric_graph_data_list_from_smiles_and_labels

print('utils.py loaded from Drive — using your exact implementations.')


utils.py loaded from Drive — using your exact implementations.


## 5 · Configuration

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
DATA_CSV       = f'{DRIVE_ROOT}/tox21_dataset.csv'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints_tox21'
RESULTS_DIR    = f'{DRIVE_ROOT}/results_tox21'

# ── Optuna toggle ─────────────────────────────────────────────────────────────
# Set RUN_OPTUNA = True to search for hyperparameters first.
RUN_OPTUNA   = False
N_TRIALS     = 20      # number of Optuna trials (each trains 50 epochs)

# ── Known-best params (used when RUN_OPTUNA = False, or as Optuna fallback) ──
KNOWN_LAYER_TYPES = ['sage', 'gat', 'gin']
KNOWN_HIDDEN = 400
KNOWN_DROPOUT = 0.25
KNOWN_LR = 0.0002
KNOWN_WD = 1e-4
# ── 30-run experiment settings ────────────────────────────────────────────────
N_RUNS = 10
MAX_EPOCHS = 500
EVAL_EVERY = 10
PATIENCE = 20
NUM_GRAPHS_PER_BATCH = 64
USE_SCAFFOLD_SPLIT = True
GLOBAL_SEED = 43
POS_WEIGHT = 6.0      # BCEWithLogitsLoss weight for positive class

print('Config OK')
print(f'  RUN_OPTUNA={RUN_OPTUNA}  N_TRIALS={N_TRIALS}')
print(f'  scaffold_split={USE_SCAFFOLD_SPLIT}  N_RUNS={N_RUNS}  MAX_EPOCHS={MAX_EPOCHS}')


Config OK
  RUN_OPTUNA=False  N_TRIALS=20
  scaffold_split=True  N_RUNS=10  MAX_EPOCHS=500


## 6 · Load dataset & build splits

In [ ]:
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

df = pd.read_csv(DATA_CSV)
df_task = df['smiles'].dropna().reset_index(drop=True)
print(f'Compounds after dropna: {len(df_task)}')
print(df_task['SR-ARE'].value_counts())


task_cols = [col for col in df.columns if col not in ['smiles', 'mol_id']]

X_smiles, y_labels = [], []
for _, row in df.iterrows():
    smi = row['smiles']
    if Chem.MolFromSmiles(smi) is not None:
        X_smiles.append(smi)
        y_labels.append(row[task_cols].values.astype(float))

y_labels = np.array(y_labels)
print(f'Valid SMILES: {len(X_smiles)} / {len(X_smiles_raw)}')

print('Building molecular graphs (1-2 min)...')
data_list = smiles_to_graph_list(X_smiles, y_labels)
assert len(data_list) == len(X_smiles), "Length mismatch after graph building!"
print(f'Graph list: {len(data_list)} molecules')

# ── Fixed split — same across all 30 runs ────────────────────────────────────
if USE_SCAFFOLD_SPLIT:
    train_idx, val_idx, test_idx = scaffold_split(X_smiles, seed=GLOBAL_SEED)
else:<f
    all_idx = list(range(len(data_list)))
    train_idx, temp = train_test_split(all_idx, test_size=0.2, random_state=GLOBAL_SEED)
    val_idx, test_idx = train_test_split(temp,   test_size=0.5, random_state=GLOBAL_SEED)
    print(f'Random split → train:{len(train_idx)} val:{len(val_idx)} test:{len(test_idx)}')

val_loader  = DataLoader([data_list[i] for i in val_idx],
                         batch_size=NUM_GRAPHS_PER_BATCH, shuffle=False, drop_last=False)
test_loader = DataLoader([data_list[i] for i in test_idx],
                         batch_size=NUM_GRAPHS_PER_BATCH, shuffle=False, drop_last=False)
trainval_data = [data_list[i] for i in list(train_idx) + list(val_idx)]

print('Data splits ready.')


Compounds after dropna: 5832
SR-ARE
0.0    4890
1.0     942
Name: count, dtype: int64


[20:14:44] Explicit valence for atom # 8 Al, 6, is greater than permitted
[20:14:44] Explicit valence for atom # 3 Al, 6, is greater than permitted
[20:14:44] Explicit valence for atom # 4 Al, 6, is greater than permitted
[20:14:44] Explicit valence for atom # 4 Al, 6, is greater than permitted
[20:14:44] Explicit valence for atom # 9 Al, 6, is greater than permitted
[20:14:44] Explicit valence for atom # 5 Al, 6, is greater than permitted
[20:14:44] Explicit valence for atom # 16 Al, 6, is greater than permitted


Valid SMILES: 5825 / 5832
Building molecular graphs (1-2 min)...
Graph list: 5825 molecules
Scaffold split → train: 3495 (60.0%), val: 1165 (20.0%), test: 1165 (20.0%)
Data splits ready.


## 7 · GNN model

In [ ]:
class GNN(torch.nn.Module):
    def __init__(self, layer_types, hidden_dim, dropout):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        in_dim = 79  # fixed by get_atom_features in utils.py
        for lt in layer_types:
            if lt == 'gcn':
                self.convs.append(GCNConv(in_dim, hidden_dim))
            elif lt == 'gat':
                self.convs.append(GATConv(in_dim, hidden_dim))
            elif lt == 'gin':
                mlp = nn.Sequential(
                    nn.Linear(in_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(),
                    nn.Linear(hidden_dim, hidden_dim), nn.ReLU())
                self.convs.append(GINConv(mlp, eps=0.00005, train_eps=True))
            elif lt == 'sage':
                self.convs.append(SAGEConv(in_dim, hidden_dim))
            else:
                raise ValueError(f'Unknown layer type: {lt}')
            in_dim = hidden_dim
        self.drop = nn.Dropout(p=dropout)
        self.out  = Linear(hidden_dim * 2, 1)

    def forward(self, x, edge_index, batch_index):
        for conv in self.convs:
            x = conv(x, edge_index)
            x = torch.tanh(x)
        x = self.drop(x)
        x = torch.cat([gmp(x, batch_index), gap(x, batch_index)], dim=1)
        return self.out(x)


def evaluate_auc(model, loader, device):
    """ROC-AUC with raw sigmoid probabilities — correct for AUC (no thresholding)."""
    model.eval()
    y_true_all, y_score_all = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch.x.float(), batch.edge_index, batch.batch).squeeze()
            probs  = torch.sigmoid(logits)
            y_true_all.append(batch.y.cpu().numpy())
            y_score_all.append(probs.cpu().numpy())
    return roc_auc_score(np.concatenate(y_true_all), np.concatenate(y_score_all))


print('GNN defined.')


GNN defined.


## 8 · Optuna hyperparameter search

In [ ]:
pos_weight_tensor = torch.FloatTensor([POS_WEIGHT]).to(device)

if RUN_OPTUNA:
    # Build a fixed train loader for Optuna trials
    optuna_train_loader = DataLoader(
        [data_list[i] for i in train_idx],
        batch_size=NUM_GRAPHS_PER_BATCH, shuffle=True, drop_last=True)

    def objective(trial):
        layer_types = [trial.suggest_categorical(f'layer_{i}', ['gcn','gat','gin','sage'])
                       for i in range(3)]
        hidden_dim  = trial.suggest_int('hidden_dim', 64, 256, step=32)
        dropout     = trial.suggest_float('dropout', 0.0, 0.5)
        lr          = trial.suggest_float('lr', 1e-5, 1e-3, log=True)

        m   = GNN(layer_types, hidden_dim, dropout).to(device)
        opt = torch.optim.Adam(m.parameters(), lr=lr)
        crit = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

        best_val, patience_ctr = 0.0, 0
        for epoch in range(100):
            m.train()
            for batch in optuna_train_loader:
                batch = batch.to(device)
                opt.zero_grad()
                pred = m(batch.x.float(), batch.edge_index, batch.batch).squeeze()
                crit(pred, batch.y).backward()
                opt.step()

            val_auc = evaluate_auc(m, val_loader, device)
            trial.report(val_auc, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

            if val_auc > best_val:
                best_val, patience_ctr = val_auc, 0
            else:
                patience_ctr += 1
                if patience_ctr >= 5:
                    break
        return best_val

    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
    study  = optuna.create_study(direction='maximize', pruner=pruner)

    print(f'Running Optuna ({N_TRIALS} trials)...')
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    best_p      = study.best_trial.params
    best_layer_types = [best_p[f'layer_{i}'] for i in range(3)]
    best_hidden      = best_p['hidden_dim']
    best_dropout     = round_to_4(best_p['dropout'])
    best_lr          = round_to_4(best_p['lr'])

    print(f'\nBest trial AUC: {study.best_trial.value:.4f}')
    print(f'  layers={best_layer_types}  hidden={best_hidden}  dropout={best_dropout}  lr={best_lr}')

    # Save Optuna results to Drive
    from datetime import datetime
    today = datetime.today().strftime('%Y-%m-%d')
    optune_filename =f'optuna_results_{today}.txt'
    optuna_path = os.path.join(RESULTS_DIR, optune_filename)
    with open(optuna_path, 'w') as f:
        f.write('Optuna Study Results\n' + '='*60 + '\n')
        f.write(f'Best AUC: {study.best_trial.value:.6f}\n')
        f.write(f'Best params:\n')
        for k, v in study.best_trial.params.items():
            f.write(f'  {k}: {round_to_4(v)}\n')
        f.write('\nAll trials (sorted):\n' + '='*60 + '\n')
        for rank, t in enumerate(sorted(
            [t for t in study.trials if t.value is not None],
            key=lambda t: t.value, reverse=True), 1):
            f.write(f'{rank}. Trial #{t.number}: AUC={t.value:.6f} | {t.params}\n')
    print(f'Optuna results saved to {optuna_path}')

else:
    best_layer_types = KNOWN_LAYER_TYPES
    best_hidden      = KNOWN_HIDDEN
    best_dropout     = KNOWN_DROPOUT
    best_lr          = KNOWN_LR
    print(f'Skipping Optuna — using known-best params:')
    print(f'  layers={best_layer_types}  hidden={best_hidden}  dropout={best_dropout}  lr={best_lr}')


Skipping Optuna — using known-best params:
  layers=['sage', 'gat', 'gin']  hidden=400  dropout=0.25  lr=0.0002


## 9 · Save helpers

In [ ]:
def append_run_result(results_dir, name, run_id, best_auc, mod_auc, auc_history):
  """Write per-run metrics (best_auc, mod_auc) to the model txt log and master CSV."""
  os.makedirs(results_dir, exist_ok=True)

  # 1. Per-model text log
  txt_path = os.path.join(results_dir, f'{name}.txt')
  with open(txt_path, 'a') as f:
    f.write(
        f'run:{run_id:02d}  best_auc:{best_auc:.6f}  mod_auc:{mod_auc:.6f}\n'
    )

  # 2. Master CSV logging
  csv_path = os.path.join(results_dir, 'all_runs_summary.csv')
  file_exists = os.path.exists(csv_path)

  # Format history values cleanly
  formatted_auc_history = [f'{auc:.6f}' for auc in auc_history]

  with open(csv_path, 'a', newline='') as f:
    writer = csv.writer(f)

    # If the file is new, include 'mod_auc' in the header
    if not file_exists:
      ep_cols = [f'auc_step_{i+1}' for i in range(len(auc_history))]
      writer.writerow(['model', 'run', 'best_auc', 'mod_auc'] + ep_cols)

    # Write the metrics row
    writer.writerow(
        [name, run_id, f'{best_auc:.6f}', f'{mod_auc:.6f}']
        + formatted_auc_history
    )


## 10 · 30-Run experiment


In [ ]:
NAME      = '_'.join(best_layer_types)
ckpt_dir  = os.path.join(CHECKPOINT_DIR, NAME)
model_dir = os.path.join(CHECKPOINT_DIR, NAME + '_best')

print(f'Model name : {NAME}')
print(f'Checkpoints: {ckpt_dir}')
print(f'Results    : {RESULTS_DIR}')
print()

# Build PyG DataLoaders strictly from train_idx (prevents data leakage)
train_data = [data_list[i] for i in train_idx]

all_run_aucs     = []
experiment_start = time.time()

for run in range(N_RUNS):
    run_seed = run   # seeds 0 … 29
    torch.manual_seed(run_seed)
    np.random.seed(run_seed)
    random.seed(run_seed)

    model  = GNN(best_layer_types, best_hidden, best_dropout).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=best_lr, weight_decay=KNOWN_WD)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    train_loader = DataLoader(
        train_data,
        batch_size=NUM_GRAPHS_PER_BATCH,
        shuffle=True,
        drop_last=True,
        generator=torch.Generator().manual_seed(run_seed),
    )

    best_val_auc = 0.0
    mod_auc      = 0.0
    patience_ctr = 0
    auc_history  = []
    run_start    = time.time()

    for epoch in range(MAX_EPOCHS):
        # Train step
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            pred = model(batch.x.float(), batch.edge_index, batch.batch).squeeze()
            criterion(pred, batch.y).backward()
            optimizer.step()

        # Evaluate on Validation set for Early Stopping
        if (epoch + 1) % EVAL_EVERY == 0:
            val_auc = evaluate_auc(model, val_loader, device)
            auc_history.append(round(val_auc, 6))
            mod_auc = val_auc  # Track the current validation checkpoint score

            if val_auc > best_val_auc:
                best_val_auc = val_auc
                patience_ctr = 0
                state = {
                    'epoch':      epoch + 1,
                    'state_dict': model.state_dict(),
                    'optimizer':  optimizer.state_dict(),
                    'val_auc':    best_val_auc,
                    'layers':     best_layer_types,
                    'hidden':     best_hidden,
                    'dropout':    best_dropout,
                    'lr':         best_lr,

                }
                save_ckp(state, True, ckpt_dir, model_dir, f'model_{NAME}_run{run:02d}_tox21.pt', f'best_model_{NAME}_run{run:02d}_tox21.pt')
            else:
                patience_ctr += 1

            if patience_ctr >= PATIENCE:
                print(f'  Run {run:02d} | Early stop @ epoch {epoch+1} | best Val AUC {best_val_auc:.4f}')
                break

    # Load the best saved checkpoint to evaluate strictly on Test Set
    best_ckpt_path = os.path.join(model_dir, f'model_{NAME}_run{run:02d}_tox21.pt')
    if os.path.exists(best_ckpt_path):
        model, optimizer, _ = load_ckp(best_ckpt_path, model, optimizer)

    test_auc = evaluate_auc(model, test_loader, device)
    all_run_aucs.append(test_auc)

    # Append run results: best_auc (test set), mod_auc (val set trigger/final), and history
    append_run_result(RESULTS_DIR, NAME, run, test_auc, mod_auc, auc_history)

    run_mins   = (time.time() - run_start) / 60
    total_mins = (time.time() - experiment_start) / 60
    print(f'Run {run:02d}/29 | Test AUC {test_auc:.4f} | Val AUC {best_val_auc:.4f} | {run_mins:.1f}m/run | total {total_mins:.1f}m')

print(f'\n{"="*50}')
print(f'EXPERIMENT COMPLETE — {NAME}')
print(f'  Median Test AUC : {np.median(all_run_aucs):.4f}')
print(f'  Std  Test AUC : {np.std(all_run_aucs):.4f}')
print(f'  Min  Test AUC : {np.min(all_run_aucs):.4f}')
print(f'  Max  Test AUC : {np.max(all_run_aucs):.4f}')
print(f'  Results      : {RESULTS_DIR}')

Model name : sage_gat_gin
Checkpoints: /content/drive/MyDrive/AUA/Thesis/code/HybridGNN/TOX/checkpoints_tox21/sage_gat_gin
Results    : /content/drive/MyDrive/AUA/Thesis/code/HybridGNN/TOX/results_tox21

  Run 00 | Early stop @ epoch 230 | best Val AUC 0.7925
Run 00/29 | Test AUC 0.7432 | Val AUC 0.7925 | 2.4m/run | total 2.4m
Run 01/29 | Test AUC 0.7476 | Val AUC 0.8178 | 5.3m/run | total 7.7m
  Run 02 | Early stop @ epoch 230 | best Val AUC 0.8014
Run 02/29 | Test AUC 0.7602 | Val AUC 0.8014 | 2.5m/run | total 10.1m
  Run 03 | Early stop @ epoch 250 | best Val AUC 0.8010
Run 03/29 | Test AUC 0.7561 | Val AUC 0.8010 | 2.6m/run | total 12.8m
  Run 04 | Early stop @ epoch 250 | best Val AUC 0.7983
Run 04/29 | Test AUC 0.7568 | Val AUC 0.7983 | 2.6m/run | total 15.4m
  Run 05 | Early stop @ epoch 270 | best Val AUC 0.7858
Run 05/29 | Test AUC 0.7294 | Val AUC 0.7858 | 2.8m/run | total 18.2m
  Run 06 | Early stop @ epoch 300 | best Val AUC 0.8003
Run 06/29 | Test AUC 0.7545 | Val AUC 0.80

KeyboardInterrupt: 

## 11 · Statistical significance
Run after you have 30-run results for all 4 baselines. Paste the AUC lists below.

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.stats as stats

# Ensure RESULTS_DIR and NAME match your notebook variables
csv_path = os.path.join(RESULTS_DIR, 'all_runs_summary.csv')

if os.path.exists(csv_path):
    df_results = pd.read_csv(csv_path)

    # Extract hybrid model scores using mod_auc
    hybrid_scores = df_results[df_results['model'] == NAME]['mod_auc'].values
    print(f'Hybrid ({NAME}): mean={np.median(hybrid_scores):.4f} ± {np.std(hybrid_scores):.4f}\n')

    header = f'{"Model":<12}  {"Mean":>7}  {"Std":>7}  {"p-value":>12}  {"Significant?":>13}'
    print(header)
    print('-' * len(header))

    models = df_results['model'].unique()
    for m in models:
        if m == NAME:
            continue
        baucs = df_results[df_results['model'] == m]['mod_auc'].values
        if len(baucs) < 2:
            continue
        _, p = stats.mannwhitneyu(hybrid_scores, baucs, alternative='greater')
        sig = 'YES p<0.05' if p < 0.05 else 'no'
        print(f'{m:<12}  {np.median(baucs):>7.4f}  {np.std(baucs):>7.4f}  {p:>12.2e}  {sig:>13}')
else:
    print(f"Summary CSV not found at {csv_path}. Run Cell 10 first.")

## 12 · Results plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# left: boxplot + strip
sns.boxplot( y=all_run_aucs, ax=axes[0], color='steelblue', width=0.4)
sns.stripplot(y=all_run_aucs, ax=axes[0], color='navy', alpha=0.5, jitter=True)
axes[0].axhline(np.median(all_run_aucs), color='red', ls='--',
                label=f'median={np.median(all_run_aucs):.4f}±{np.std(all_run_aucs):.4f}')
axes[0].set_title(f'30-run AUC distribution\n{NAME}')
axes[0].set_ylabel('ROC-AUC')
axes[0].legend()

# right: per-epoch AUC for every run
summary_df  = pd.read_csv(os.path.join(RESULTS_DIR, 'all_runs_summary.csv'))
hybrid_rows = summary_df[summary_df['model'] == NAME]
auc_cols    = [c for c in summary_df.columns if c.startswith('auc_ep')]
for _, row in hybrid_rows.iterrows():
    axes[1].plot(range(len(auc_cols)), row[auc_cols].values, alpha=0.25, color='steelblue')
axes[1].set_xlabel(f'Eval checkpoint (every {EVAL_EVERY} epochs)')
axes[1].set_ylabel('Test ROC-AUC')
axes[1].set_title('AUC over training — all 30 runs')

plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, f'{NAME}_results.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Saved to {plot_path}')


## 13 · Load a saved checkpoint (optional)
Use this cell to reload the best model from any run — e.g. to run inference or inspect weights.
The checkpoint saved in Cell 10 contains the full model state including which layers were used.


In [ ]:
torch.serialization.add_safe_globals([np._core.multiarray.scalar])

# ── Edit run_id to load whichever run you want ────────────────────────────────
run_id       = 0
load_name    = NAME   # or hardcode e.g. 'sage_gin_sage'
load_model_dir = os.path.join(CHECKPOINT_DIR, load_name + '_best')
ckpt_path    = os.path.join(load_model_dir,
                            f'model_{load_name}_run{run_id:02d}_tox21.pt')

# Reconstruct the model with the same architecture
loaded_model = GNN(best_layer_types, best_hidden, best_dropout).to(device)
loaded_opt   = torch.optim.Adam(loaded_model.parameters(), lr=best_lr)

loaded_model, loaded_opt, start_epoch = load_ckp(ckpt_path, loaded_model, loaded_opt)

# Verify
auc = evaluate_auc(loaded_model, test_loader, device)
print(f'Loaded run {run_id} checkpoint (trained to epoch {start_epoch})')
print(f'Test AUC on reload: {auc:.4f}')